# Human-in-the-Loop: Pre-Actie Poorten, Risiconiveaus en Audit Logging

De README voor deze les introduceert Human-in-the-Loop met een korte snippet die de gebruiker vraagt om `APPROVE` of `REJECT` nadat de agent al een reactie heeft geproduceerd. Dat patroon is een prima startpunt, maar productieve HITL-implementaties hebben meestal drie extra onderdelen nodig:

1. Een **pre-actie poort** die **voor** de agent een risicovolle stap uitvoert, zodat kosten, onomkeerbaarheid en latentie onder controle blijven.
2. **Risiconiveaus**, zodat acties met laag risico automatisch worden uitgevoerd, acties met middelhoog risico in batches worden goedgekeurd, en alleen acties met hoog risico door een mens worden geblokkeerd.
3. Een **auditlog plus revisielus**, zodat elke poortbeslissing als JSONL wordt geregistreerd, en een afwijzing de agent heropent met een gestructureerde reden in plaats van alleen `Revising...` af te drukken.

Deze notebook bouwt elk van deze onderdelen op bovenop dezelfde basiselementen als `06-system-message-framework.ipynb`. Hij draait end-to-end in `DEMO_MODE = True` (geen interactieve invoer nodig) of met echte `input()` prompts wanneer `DEMO_MODE = False`. Opmerking: in DEMO_MODE is het herhalen van het derde doel gescript zodat de loop-mechanica end-to-end zichtbaar is. Echte revision-driven herclassificatie vereist `DEMO_MODE = False` en een operator.

**Buiten scope (behandeld in andere lessen):** authenticatie en toegangscontrole (Les 06 README bedreiging 2), tool-call middleware (Les 14 MAF deep dive), multi-agent debatpatronen.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Patroon 1: Pre-actietoegang

De HITL-code uit de README roept eerst de agent aan en vraagt de gebruiker vervolgens de uitvoer goed te keuren. Dat is een **post-actiestroom**. De agent heeft al uitgevoerd, dus de LLM-kosten zijn al betaald en elke nevenwerking (verzonden e-mail, geschreven database-rij, geplaatste opmerking) heeft al plaatsgevonden.

Een **pre-actiestroom** plaatst het toegangspunt voordat de agent de risicovolle stap uitvoert. De agent stelt de actie voor, het toegangspunt bepaalt of deze uitgevoerd wordt, en alleen bij goedkeuring vindt de nevenwerking plaats.

| Aspect | Post-actiegoedkeuring (README-code) | Pre-actietoegang (deze notebook) |
|---|---|---|
| Wanneer wordt goedkeuring uitgevoerd? | Nadat de agent output heeft geproduceerd | Voordat een nevenwerking wordt uitgevoerd |
| LLM-kosten bij afwijzing | Al betaald | Alleen betaald voor het voorstel, niet voor de actie |
| Onomkeerbare nevenwerkingen | Mogelijk (de actie heeft al plaatsgevonden) | Voorkomen |
| Audit duidelijkheid | Goedkeuring is een print statement | Goedkeuring is een JSON record met tijdstempel, actie, reden |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Patroon 2: Risicoklassen

Niet elke actie heeft menselijke goedkeuring nodig. Een alleen-lezen opvraging tegen een openbare API heeft andere belangen dan het versturen van een klant-e-mail. Beide op dezelfde manier behandelen verspilt de aandacht van de operator en vertraagt de agent.

Een eenvoudig 3-laags model:

| Laag | Voorbeelden | Goedkeuringsproces |
|---|---|---|
| `laag` (alleen-lezen) | Zoek een kennisbank op, zoek vluchtopties, haal een openbare webpagina op | Automatisch uitvoeren, gelogd voor audit |
| `medium` (goedkope wijziging) | Cache een resultaat, verhoog een teller, plan een herinnering | Automatisch uitvoeren, maar dagelijks gebundeld gecontroleerd |
| `hoog` (externe of onomkeerbare acties) | Verstuur een e-mail, incasseer een kaart, plaats in een openbare kanaal | Wacht op menselijke goedkeuring |

Dit is één indeling. Productiesystemen gebruiken vaak meer gedetailleerde lagen (bijv. AWS IAM permissieniveaus, rolgebaseerde toegangslevels). De 3-laags versie hieronder is de kleinste nuttige versie voor een agent die alleen-lezen en zij-effect acties mixt.

De onderstaande classifier gebruikt sleutelwoordheuristieken zodat de demo deterministisch en goedkoop blijft. In een productiesysteem zou je dit vervangen door een geleerde classifier of een beleidsmotor.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Patroon 3: Auditlog en revisielus

Een `print("Response approved.")` is geen auditlog. Voor vertrouwen moet elke poortbeslissing worden vastgelegd als een gestructureerd gebeurtenis die je later kunt opvragen, afspelen of koppelen aan een incidentbeoordeling.

Twee onderdelen:

1. **Append-only JSONL.** Eén regel per beslissing, met tijdstempel, actie, niveau, beslissing, reden. Makkelijk te doorzoeken, makkelijk om later naar een echte logopslag te sturen.
2. **Revisielus bij afwijzing.** Wanneer de poort `deny` teruggeeft, geeft de agent zichzelf opnieuw een prompt met de afwijzingsreden in de context, zodat het volgende voorstel het probleem kan vermijden.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Aanvullende bronnen

Verschillende andere openbare projecten implementeren variaties van deze HITL-patronen. Vergelijk de benaderingen om te zien wat bij jouw stack past:

- **LangChain** human-in-the-loop tool wrappers ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): plug-in tool wrappers die de uitvoering pauzeren voor menselijke input.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ heeft dit herverdeeld): gebruikt een agentrol specifiek om de mens te vertegenwoordigen in multi-agentgesprekken.
- **Semantic Kernel** functiefilters ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware die rond elke functieaanroep draait, geschikt voor gate-logica.

Elk project behandelt de drie subpatronen anders: LangChain wikkelt ze in als tools, AutoGen gebruikt een agentrol, Semantic Kernel gebruikt middleware-filters. Lees één of twee implementaties volledig door voordat je een ontwerp kiest voor je eigen agent.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
